## Limpieza de 'DF_CRONOFINISHER_SUCIO.csv'

Cargamos la tabla de carreras/modalidades (una fila por cada modalidad de cada evento) y le hacemos una primera inspección antes de limpiarla, siguiendo el mismo proceso que en buscametas/xipgroc/carreirasgalegas/ccnorte.

In [1]:
from pathlib import Path
import pandas as pd

CSV_PATH = Path("../../data/raw/cronofinisher/DF_CRONOFINISHER_SUCIO.csv")
curses = pd.read_csv(CSV_PATH, encoding="utf-8-sig")

print("Filas x columnas:", curses.shape)
print()
print(curses.dtypes)
curses.head()

Filas x columnas: (1406, 18)

event_id                       float64
nom_cursa                       object
data                            object
lloc                            object
modalitat_nom                   object
modalitat_codi                  object
estat_esdeveniment              object
error_detall                    object
total_classificats             float64
classificat_h                  float64
classificat_d                  float64
classificat_total              float64
inscrits                       float64
DNF_total                      float64
DSQ_total                      float64
DNS_total                      float64
esport                          object
classificat_sexe_desconegut    float64
dtype: object


,event_id,nom_cursa,data,lloc,modalitat_nom,modalitat_codi,estat_esdeveniment,error_detall,total_classificats,classificat_h,classificat_d,classificat_total,inscrits,DNF_total,DSQ_total,DNS_total,esport,classificat_sexe_desconegut
0,719.0,IX CARRERA DE ROCHE POR EL SINDROME DE RETT,2026-08-09,CONIL DE LA FRONTERA,5K -,5k,ok,NaN,206.0,0.0,0.0,206.0,NaN,NaN,NaN,NaN,Carrera,206.0
1,719.0,IX CARRERA DE ROCHE POR EL SINDROME DE RETT,2026-08-09,CONIL DE LA FRONTERA,10 K,10k,ok,NaN,256.0,0.0,0.0,256.0,NaN,NaN,NaN,NaN,Carrera,256.0
2,719.0,IX CARRERA DE ROCHE POR EL SINDROME DE RETT,2026-08-09,CONIL DE LA FRONTERA,INFANTIL CADETE -,infantilcadete,ok,NaN,16.0,0.0,0.0,16.0,NaN,NaN,NaN,NaN,Carrera,16.0
3,719.0,IX CARRERA DE ROCHE POR EL SINDROME DE RETT,2026-08-09,CONIL DE LA FRONTERA,ALEVIN -,alevin,ok,NaN,22.0,0.0,0.0,22.0,NaN,NaN,NaN,NaN,Carrera,22.0
4,719.0,IX CARRERA DE ROCHE POR EL SINDROME DE RETT,2026-08-09,CONIL DE LA FRONTERA,BENJAMIN -,benjamin,ok,NaN,19.0,0.0,0.0,19.0,NaN,NaN,NaN,NaN,Carrera,19.0


In [2]:
# Diagnóstico antes de limpiar: nulos, duplicados, estado del scraping y
# rango de fechas. A diferencia de las otras fuentes, aquí cada evento
# lleva su propio "estat_esdeveniment" (el scraper marca si pudo sacar
# resultados o no), así que lo comprobamos antes de nada.
print("Valores nulos por columna:")
print(curses.isna().sum())
print()

print("Filas completamente duplicadas:", curses.duplicated().sum())
print("Filas duplicadas por (event_id, modalitat_codi):",
      curses.duplicated(subset=["event_id", "modalitat_codi"]).sum())
print()

print("Rango de fechas (texto):", curses["data"].min(), "->", curses["data"].max())
print()

print("estat_esdeveniment:")
print(curses["estat_esdeveniment"].value_counts())

Valores nulos por columna:
event_id                         92
nom_cursa                         0
data                              0
lloc                              0
modalitat_nom                    93
modalitat_codi                   93
estat_esdeveniment                0
error_detall                   1388
total_classificats              111
classificat_h                   111
classificat_d                   111
classificat_total               111
inscrits                       1391
DNF_total                      1391
DSQ_total                      1394
DNS_total                      1339
esport                            0
classificat_sexe_desconegut     111
dtype: int64

Filas completamente duplicadas: 0
Filas duplicadas por (event_id, modalitat_codi): 91

Rango de fechas (texto): 2015-01-11 -> 2026-08-09

estat_esdeveniment:
estat_esdeveniment
ok                 1295
sense_resultats      93
llista_error         18
Name: count, dtype: int64


In [3]:
# Nos quedamos solo con eventos con resultados reales ("ok") — "sense_resultats"
# y "llista_error" son eventos donde el scraper no pudo sacar nada (y por
# eso event_id/modalitat_nom/los recuentos salen NaN). Después, quitamos
# duplicados exactos igual que en xipgroc/carreirasgalegas.
antes = len(curses)
curses = curses[curses["estat_esdeveniment"] == "ok"].reset_index(drop=True)
print(f"Filas sin resultados descartadas: {antes - len(curses)} ({antes} -> {len(curses)})")

antes = len(curses)
curses = curses.drop_duplicates().reset_index(drop=True)
print(f"{antes - len(curses)} filas duplicadas eliminadas ({antes} -> {len(curses)})")

Filas sin resultados descartadas: 111 (1406 -> 1295)
0 filas duplicadas eliminadas (1295 -> 1295)


In [4]:
# Limpieza: nos quedamos con las columnas que interesan, renombradas.
# "lloc" (municipio) sí viene en esta fuente, igual que en carreirasgalegas
# — no hace falta geocodificarlo desde el nombre de la carrera como en
# xipgroc. finisher_d/finisher_h/finisher_desconocido: mismo nombre que en
# el resto de fuentes (antes classificat_d/classificat_h/
# classificat_sexe_desconegut). event_id -> id.
curses_limpio = curses[
    ["nom_cursa", "data", "lloc", "esport", "modalitat_nom",
     "classificat_d", "classificat_h", "classificat_sexe_desconegut", "event_id"]
].rename(columns={
    "nom_cursa": "nombre_carrera",
    "data": "fecha",
    "lloc": "municipio",
    "modalitat_nom": "modalidad",
    "classificat_d": "finisher_d",
    "classificat_h": "finisher_h",
    "classificat_sexe_desconegut": "finisher_desconocido",
    "event_id": "id",
})

curses_limpio["fecha"] = pd.to_datetime(curses_limpio["fecha"])
curses_limpio["id"] = curses_limpio["id"].astype(int)
curses_limpio[["finisher_d", "finisher_h", "finisher_desconocido"]] = (
    curses_limpio[["finisher_d", "finisher_h", "finisher_desconocido"]].fillna(0).astype(int)
)

print(curses_limpio.shape)
curses_limpio.head()

(1295, 9)


,nombre_carrera,fecha,municipio,esport,modalidad,finisher_d,finisher_h,finisher_desconocido,id
0,IX CARRERA DE ROCHE POR EL SINDROME DE RETT,2026-08-09,CONIL DE LA FRONTERA,Carrera,5K -,0,0,206,719
1,IX CARRERA DE ROCHE POR EL SINDROME DE RETT,2026-08-09,CONIL DE LA FRONTERA,Carrera,10 K,0,0,256,719
2,IX CARRERA DE ROCHE POR EL SINDROME DE RETT,2026-08-09,CONIL DE LA FRONTERA,Carrera,INFANTIL CADETE -,0,0,16,719
3,IX CARRERA DE ROCHE POR EL SINDROME DE RETT,2026-08-09,CONIL DE LA FRONTERA,Carrera,ALEVIN -,0,0,22,719
4,IX CARRERA DE ROCHE POR EL SINDROME DE RETT,2026-08-09,CONIL DE LA FRONTERA,Carrera,BENJAMIN -,0,0,19,719


In [5]:
# A diferencia de buscametas/carreirasgalegas/ccnorte, aquí la disciplina
# ya viene dada directamente por "esport" (14 valores posibles) — igual
# que "Modalitat" en xipgroc — así que no hace falta clasificarla por
# palabras clave, solo mapearla al mismo esquema de categorías.
# "Travesía" (natación), "Carrera de obstáculos", "Carrera Híbrida",
# "Canicross" y "Fútbol sala" no encajan en ninguna categoría existente,
# así que van a "Otros".
_ESPORT_A_TIPO = {
    "Carrera": "road running",
    "Carrera por montaña": "trail running",
    "Trail": "trail running",
    "Ciclismo": "Ciclismo y btt",
    "Triatlón": "Multidisciplina",
    "Duatlón": "Multidisciplina",
    "Duatlón cros": "Multidisciplina",
    "Acuatlón": "Multidisciplina",
    "Senderismo": "marcha",
    "Travesía": "Otros",
    "Carrera de obstáculos": "Otros",
    "Carrera Híbrida": "Otros",
    "Canicross": "Otros",
    "Fútbol sala": "Otros",
}
curses_limpio["tipo_modalidad"] = curses_limpio["esport"].map(_ESPORT_A_TIPO)
curses_limpio = curses_limpio.drop(columns=["esport"])

print(curses_limpio["tipo_modalidad"].value_counts())

tipo_modalidad
road running       736
trail running      236
Ciclismo y btt     144
Otros              102
Multidisciplina     77
Name: count, dtype: int64


In [6]:
# "modalidad" aquí es el campo más sucio de las 5 fuentes (661 valores
# únicos: distancias, categorías de edad, y sobre todo texto logístico de
# cronometraje — "CON PARCIALES", "CON PUNTO INTERMEDIO", "LOCAL"...).
# Extraemos la distancia donde se pueda: K/KM, M/M. (aquí el punto es
# separador de miles, no decimal — "3.300 M." = 3300 m = 3,3 km, al
# revés que en el resto de fuentes), maratón/media maratón/milla con
# distancia oficial, y como último recurso, códigos puramente numéricos
# sueltos ("1500", "3000"...) que aquí son metros (habituales en pruebas
# de pista/natación).
import re

_km = curses_limpio["modalidad"].str.extract(r"(\d+(?:[.,]\d+)?)\s*km?\b", flags=re.IGNORECASE)[0]
distancia_km = _km.str.replace(",", ".", regex=False).astype(float)

_m = curses_limpio["modalidad"].str.extract(r"(\d{1,3}(?:\.\d{3})*)\s*m\.?\b", flags=re.IGNORECASE)[0]
distancia_m = _m.str.replace(".", "", regex=False).astype(float) / 1000

curses_limpio["distancia"] = distancia_km
_falta = curses_limpio["distancia"].isna()
curses_limpio.loc[_falta, "distancia"] = distancia_m[_falta]

_falta = curses_limpio["distancia"].isna()
_es_media = curses_limpio["modalidad"].str.contains(r"media\s*marat", case=False, regex=True, na=False)
_es_marat = curses_limpio["modalidad"].str.contains(r"marat", case=False, regex=True, na=False)
_es_milla = curses_limpio["modalidad"].str.contains(r"milla", case=False, regex=True, na=False)
curses_limpio.loc[_falta & _es_media, "distancia"] = 21.097
curses_limpio.loc[_falta & ~_es_media & _es_marat, "distancia"] = 42.195
curses_limpio.loc[_falta & _es_milla, "distancia"] = 1.609

_falta = curses_limpio["distancia"].isna()
_bare = curses_limpio["modalidad"].str.strip().str.extract(r"^(\d{2,6})$")[0]
curses_limpio.loc[_falta, "distancia"] = _bare[_falta].astype(float) / 1000

curses_limpio["distancia"] = curses_limpio["distancia"].fillna(0)

print("Filas con distancia detectada:", (curses_limpio["distancia"] != 0).sum(),
      "de", len(curses_limpio))
print()
print("Ejemplos de modalidad SIN distancia detectada (revisa si falta algún patrón):")
print(curses_limpio.loc[curses_limpio["distancia"] == 0, "modalidad"].value_counts().head(30))

Filas con distancia detectada: 193 de 1295

Ejemplos de modalidad SIN distancia detectada (revisa si falta algún patrón):
modalidad
GENERAL                   126
ALEVIN                     35
BENJAMIN                   35
ADULTOS                    32
PREBENJAMIN                25
LOCAL                      25
ABSOLUTA                   24
INFANTIL                   17
INFANTIL CADETE            14
CON PARCIALES              14
CADETE                     11
CORTA                      11
SUB10                      11
LARGA                      10
SUB12                       9
SUB8                        8
PAREJAS                     7
MINITRAIL                   7
ELITE                       7
SUB14                       6
EQUIPOS                     6
CON PUNTO INTERMEDIO        6
ALEVIN -                    5
CON PUNTOS INTERMEDIOS      5
ADULTOS -                   5
INFANTIL-CADETE             5
SUB16                       5
POPULAR                     5
BENJAMIN ALEVIN             

In [7]:
# Clasificamos el público (edad) por palabras clave, con SubXX llevando
# la edad directamente en el número (≤12 Infantil, 13-23 Cadete/Juvenil,
# igual que en xipgroc). Antes comprobamos si es una categoría especial
# ("GENUINE" = categoría inclusiva para personas con discapacidad
# intelectual, habitual en circuitos españoles) o de élite/profesional.
# Si no hay ninguna marca de edad ni es especial, asumimos
# Absoluta/General por defecto.
_EQUIPOS_PATRON = r"equipos?\b|equips?\b"

def _clasificar_publico(row):
    nombre = row["nombre_carrera"]
    texto_nombre = "" if pd.isna(nombre) else nombre.lower()
    if re.search(_EQUIPOS_PATRON, texto_nombre):
        return "Equipos"

    texto = row["modalidad"]
    t = "" if pd.isna(texto) else texto.lower().strip()

    if re.search(_EQUIPOS_PATRON, t):
        return "Equipos"

    if re.search(r"discap|invident|handbike|silla de ruedas|adaptad|genuine", t):
        return "Otros"

    if re.search(r"\belit|profesional|\bpro\b", t):
        return "Elite"

    if re.search(r"veteran|master|m[aá]ster|\bsenior\b|\bsen\b", t):
        return "Mayores/Veteranos"

    _sb = re.search(r"\bsub\s?-?(\d{1,2})(?!\d)", t)
    if _sb:
        edad = int(_sb.group(1))
        if edad <= 12:
            return "Infantil"
        if edad <= 23:
            return "Cadete/Juvenil"

    if re.search(
        r"prebenjam|benjam|alev|infant|chupet|pitufo|querubin|figurit|retaco|"
        r"biber[oó]n|menores|escolar|ni[nñ]os|a[nñ]os|queruben",
        t,
    ):
        return "Infantil"

    if re.search(r"cadete|juvenil|junior|promesa|preuniversitari", t):
        return "Cadete/Juvenil"

    return "Absoluta/General"

curses_limpio["publico"] = curses_limpio.apply(_clasificar_publico, axis=1)

print(curses_limpio["publico"].value_counts())
print()
print("Texto de modalidad clasificado como Otros (categorías especiales):")
print(curses_limpio.loc[curses_limpio["publico"] == "Otros", "modalidad"].value_counts())

publico
Absoluta/General     844
Infantil             305
Cadete/Juvenil        90
Equipos               27
Mayores/Veteranos     18
Elite                 10
Otros                  1
Name: count, dtype: int64

Texto de modalidad clasificado como Otros (categorías especiales):
modalidad
GENUINE -18    1
Name: count, dtype: int64


### Esquema común entre las 10 fuentes

Para poder comparar o concatenar directamente las tablas de buscametas, xipgroc, carreirasgalegas, ccnorte, cronofinisher, mychip, sportmaniacs, cursescat, iter5 y cruzandolameta, las 12 columnas que comparten todas van con el mismo nombre y en el mismo orden: `fuente`, `nombre_carrera`, `fecha`, `dia_semana`, `distancia`, `tipo_modalidad`, `publico`, `finisher_d`, `finisher_h`, `municipio`, `comarca`, `provincia`. `fuente` es una constante ("cronofinisher") para identificar el origen al concatenar las 10 tablas. `municipio` ya viene de la fuente (`lloc`); `comarca`/`provincia` no, así que las geocodificamos a partir de `municipio` (igual que en carreirasgalegas), sin asumir ninguna comunidad autónoma concreta — aquí hay eventos de toda España, no solo de una región. Lo que es propio solo de cronofinisher (`finisher_desconocido`, `id`, `modalidad`) va al final.

In [8]:
# Geocodificamos "comarca"/"provincia" a partir de "municipio" (nombre ya
# limpio, sin ruido, así que geocodificamos directamente sin heurística de
# extracción). Solo ~150 municipios únicos, así que es rápido. Checkpoint
# propio en cronofinisher_ubicaciones.csv. Aquí (a diferencia de
# carreirasgalegas) los eventos están repartidos por toda España
# (Andalucía, Extremadura...), así que geocodificamos sin fijar ninguna
# comunidad autónoma.
import csv
import time


def geocodificar_ubicacion_municipios(municipios, out_dir, pausa_segundos: float = 1.1):
    from geopy.geocoders import Nominatim
    from geopy.exc import GeopyError

    out_path = Path(out_dir)
    csv_ubic = out_path / "cronofinisher_ubicaciones.csv"

    cache = {}
    if csv_ubic.exists():
        prev = pd.read_csv(csv_ubic, dtype=str)
        cache = {row["municipio"]: row.to_dict() for _, row in prev.iterrows()}
        print(f"Checkpoint: {len(cache)} municipios ya geocodificados")

    geolocator = Nominatim(user_agent="cronofinisher_ubicaciones_claudia")

    municipios_unicos = list(dict.fromkeys(m for m in municipios if isinstance(m, str)))
    pendientes = [m for m in municipios_unicos if m not in cache]
    print(f"Municipios a geocodificar: {len(pendientes)} (de {len(municipios_unicos)} únicos)")

    campos = ["municipio", "comarca", "provincia"]
    write_header = not csv_ubic.exists()
    with open(csv_ubic, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=campos)
        if write_header:
            writer.writeheader()

        for i, municipio in enumerate(pendientes, 1):
            fila = {"municipio": municipio, "comarca": None, "provincia": None}
            try:
                loc = geolocator.geocode(
                    f"{municipio}, España", exactly_one=True, country_codes="es",
                    addressdetails=True, timeout=10,
                )
                if loc:
                    addr = loc.raw.get("address", {})
                    fila["comarca"] = addr.get("county")
                    fila["provincia"] = addr.get("province") or addr.get("state")
            except GeopyError as e:
                print(f"  [{municipio}] error de geocodificación: {e}")
            except Exception as e:
                print(f"  [{municipio}] ERROR inesperado: {e}")

            writer.writerow(fila)
            f.flush()
            cache[municipio] = fila

            if i % 25 == 0 or i == len(pendientes):
                print(f"  Progreso: {i}/{len(pendientes)}")
            time.sleep(pausa_segundos)

    print(f"CSV de ubicaciones: {csv_ubic.resolve()}")
    return cache


OUT_DIR = Path("../../data/raw/cronofinisher")
_ubicaciones = geocodificar_ubicacion_municipios(curses_limpio["municipio"], out_dir=OUT_DIR)
curses_limpio["comarca"] = curses_limpio["municipio"].map(lambda m: _ubicaciones.get(m, {}).get("comarca"))
curses_limpio["provincia"] = curses_limpio["municipio"].map(lambda m: _ubicaciones.get(m, {}).get("provincia"))

print("Filas con provincia:", curses_limpio["provincia"].notna().sum(), "de", len(curses_limpio))
print("Filas con comarca:", curses_limpio["comarca"].notna().sum(), "de", len(curses_limpio))
curses_limpio[["municipio", "comarca", "provincia"]].drop_duplicates().sample(15, random_state=0)

Checkpoint: 149 municipios ya geocodificados


Municipios a geocodificar: 0 (de 149 únicos)
CSV de ubicaciones: C:\Users\Clàudia Rafart\OneDrive - Prenomics\Escritorio\Altres\cronofinisher_data\cronofinisher_ubicaciones.csv
Filas con provincia: 1283 de 1295
Filas con comarca: 439 de 1295


,municipio,comarca,provincia
1239,LA PALMA DEL CONDADO,El Condado,Huelva
950,AGRUPACION MOGON,NaN,Jaén
427,HERRERA,NaN,Sevilla
632,Berlanga,El Bierzo,León
27,SAN MARTIN DEL TESORILLO,Campo de Gibraltar,Andalucía
905,PRIEGO DE CORDOBA,NaN,Córdoba
1267,El Ejido,NaN,Almería
811,PALMA DEL RIO,NaN,Córdoba
1071,Agrupación de Mogón (Villacarrillo),NaN,Jaén
710,SEVILLA,NaN,Sevilla


In [9]:
# Añadimos "fuente" (constante, para identificar el origen al concatenar
# con las otras 5 tablas) y "dia_semana" (derivado de "fecha"), y
# reordenamos las columnas para que el esquema común (fuente,
# nombre_carrera, fecha, dia_semana, distancia, tipo_modalidad, publico,
# finisher_d, finisher_h, municipio, comarca, provincia) quede igual en
# las 6 fuentes, dejando lo propio de cronofinisher (finisher_desconocido,
# id, modalidad) al final.
curses_limpio["fuente"] = "cronofinisher"

_DIAS_SEMANA = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]
curses_limpio["dia_semana"] = curses_limpio["fecha"].dt.dayofweek.map(dict(enumerate(_DIAS_SEMANA)))

curses_limpio = curses_limpio[
    ["fuente", "nombre_carrera", "fecha", "dia_semana", "distancia", "tipo_modalidad", "publico",
     "finisher_d", "finisher_h", "municipio", "comarca", "provincia",
     "finisher_desconocido", "id", "modalidad"]
]
curses_limpio.columns.tolist()

['fuente',
 'nombre_carrera',
 'fecha',
 'dia_semana',
 'distancia',
 'tipo_modalidad',
 'publico',
 'finisher_d',
 'finisher_h',
 'municipio',
 'comarca',
 'provincia',
 'finisher_desconocido',
 'id',
 'modalidad']

In [10]:
# Vista final de la tabla ya limpia y clasificada
print("Columnas:", list(curses_limpio.columns))
print("Filas x columnas:", curses_limpio.shape)
print()
print(curses_limpio.dtypes)
print()
print("Cruce tipo_modalidad x publico:")
print(pd.crosstab(curses_limpio["tipo_modalidad"], curses_limpio["publico"]))
print()
curses_limpio.sample(15)

Columnas: ['fuente', 'nombre_carrera', 'fecha', 'dia_semana', 'distancia', 'tipo_modalidad', 'publico', 'finisher_d', 'finisher_h', 'municipio', 'comarca', 'provincia', 'finisher_desconocido', 'id', 'modalidad']
Filas x columnas: (1295, 15)

fuente                          object
nombre_carrera                  object
fecha                   datetime64[ns]
dia_semana                      object
distancia                      float64
tipo_modalidad                  object
publico                         object
finisher_d                       int64
finisher_h                       int64
municipio                       object
comarca                         object
provincia                       object
finisher_desconocido             int64
id                               int64
modalidad                       object
dtype: object

Cruce tipo_modalidad x publico:
publico          Absoluta/General  Cadete/Juvenil  Elite  Equipos  Infantil  \
tipo_modalidad                                 

,fuente,nombre_carrera,fecha,dia_semana,distancia,tipo_modalidad,publico,finisher_d,finisher_h,municipio,comarca,provincia,finisher_desconocido,id,modalidad
331,cronofinisher,XLIV CROSS DEL CORDERO DE CABEZA DEL BUEY,2024-10-05,Sábado,0.0,road running,Absoluta/General,0,57,CABEZA DEL BUEY,NaN,Badajoz,0,556,ABSOLUTA
396,cronofinisher,XXXVI CROSS PEÑA DEL AGUILA PEDRO CARBALLO,2024-08-04,Domingo,0.0,road running,Infantil,0,0,VILLAR DEL REY,NaN,Badajoz,15,546,INFANTIL
289,cronofinisher,V CROSS DEL PINAR DEL REY,2024-11-17,Domingo,0.0,road running,Cadete/Juvenil,0,0,SAN ROQUE,Campo de Gibraltar,Andalucía,21,570,SUB16-SUB18
1088,cronofinisher,II CARRERA POPULAR VILLA DE RUS,2017-09-16,Sábado,0.0,road running,Absoluta/General,21,102,RUS,NaN,Jaén,0,174,ABSOLUTA
668,cronofinisher,VIII CXM MONTIZÓN - CONDADO DE JAÉN - 2022,2022-04-09,Sábado,0.0,trail running,Absoluta/General,12,24,ALDEAHERMOSA DE MONTIZON,NaN,Jaén,60,428,- CARRERA DE MONTAÑA
1222,cronofinisher,V TRIATLON TRIHERCULES CADIZ,2016-04-30,Sábado,0.0,Multidisciplina,Absoluta/General,488,0,CADIZ,Bahía de Cádiz,Andalucía,0,77,CON PARCIALES Y
703,cronofinisher,V CXM CRESTA DEL DIABLO,2021-10-31,Domingo,0.0,trail running,Absoluta/General,0,0,TORREDELCAMPO,NaN,Jaén,56,357,DIABLITO MINITRAIL
826,cronofinisher,III TRAVESIA RIO GUADALQUIVIR - CLUB NAUTICO S...,2019-10-05,Sábado,0.0,Otros,Absoluta/General,0,0,SEVILLA,NaN,Sevilla,49,328,3000 M PREMIACION
1179,cronofinisher,III CXM SIERRA CAZORLA,2016-09-11,Domingo,0.0,trail running,Cadete/Juvenil,0,0,CAZORLA,NaN,Jaén,10,84,CADETE Y JUNIOR
158,cronofinisher,XL CARRERA POPULAR FREGENAL DE LA SIERRA,2025-09-14,Domingo,0.0,road running,Cadete/Juvenil,0,0,FREGENAL DE LA SIERRA,NaN,Badajoz,4,671,SUB 14- SUB 16


In [11]:
# Guardamos la tabla ya limpia y clasificada para poder descargarla.
SALIDA = Path("../../data/processed/cronofinisher/DF_CRONOFINISHER_LIMPIO.csv")
curses_limpio.to_csv(SALIDA, index=False, encoding="utf-8-sig")
print(f"Guardado en {SALIDA}")

Guardado en C:\Users\Clàudia Rafart\OneDrive - Prenomics\Escritorio\Altres\cronofinisher_data\DF_CRONOFINISHER_LIMPIO.csv
